In [1]:
!pip install pygam

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.6/84.6 kB 4.3 MB/s eta 0:00:00


In [ ]:
# !pip install pygam

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor
from pygam import GammaGAM, InvGaussGAM, LinearGAM, s
import warnings

warnings.filterwarnings('ignore')

# 1. LOAD DATA (Adjust paths to your Kaggle environment)
train = pd.read_csv('/kaggle/input/datasets/howaimak/ey-data/train_with_CHIRPS_features_final.csv')
test  = pd.read_csv('/kaggle/input/datasets/howaimak/ey-data/test_with_CHIRPS_features_final.csv')
tmpl  = pd.read_csv('/kaggle/input/datasets/howaimak/ey-data/submission_template.csv')

TARGETS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
TA, EC, DRP = TARGETS
DRP_Q = 0.95
N_SEEDS = 30
WEIGHTS = np.array([0.6, 0.2, 0.2]) # XGB, LGB, CB

# 2. FULL FEATURE ENGINEERING (From best_model.ipynb)
def engineer_features(train, test):
    eps = 1e-6
    mndwi_clip = float(train['MNDWI'].quantile(0.99))
    
    for df in (train, test):
        df['MNDWI_clip99'] = df['MNDWI'].clip(upper=mndwi_clip)
        df['river_len_ratio_250_2000'] = df['river_len_250m'] / (df['river_len_2000m'] + eps)
        df['log_river_len_ratio_250_2000'] = np.log1p(df['river_len_ratio_250_2000'])
        df['log_dist_river_min_m'] = np.log1p(df['dist_waterway_min_m'])
        
        nir_river = df['nir'] * df['river_count_1000m']
        nir_river_clip = float(train['nir'].quantile(0.99) * train['river_count_1000m'].quantile(0.99))
        df['nir_x_river_count_1000m_clip99'] = nir_river.clip(upper=nir_river_clip)
        
        df['ppt7_x_river1000'] = df['ppt_7d_log1p'] * df['river_len_1000m']
        df['rain90_x_river1000'] = df['rain_90d_CHIRPS'] * df['river_len_1000m']
        df['NDMI_x_river250'] = df['NDMI'] * df['river_count_250m']
        df['NDMI_x_river1000'] = df['NDMI'] * df['river_count_1000m']
    return train, test

train, test = engineer_features(train, test)

# 3. FEATURE SETS: GAM vs TREES
# Trees get the full feature set
TREE_FEATS = {
    TA: ['pet', 'NDMI', 'MNDWI_clip99', 'green', 'nir', 'swir16', 'swir22', 'forest_pct_2000m', 'forest_present_2000m', 'river_count_250m', 'river_count_1000m', 'nir_x_river_count_1000m_clip99', 'soil_clay_present', 'soil_sand_present', 'soil_phh2o_present', 'ppt_7d_log1p', 'rain_days_30d', 'rain_7d_CHIRPS', 'rain_30d_CHIRPS', 'rain_90d_CHIRPS', 'ppt7_x_river1000', 'rain90_x_river1000'],
    EC: ['pet', 'NDMI', 'MNDWI_clip99', 'swir16', 'swir22', 'green', 'nir', 'forest_present_2000m', 'river_count_1000m', 'soil_clay_present', 'soil_sand_present', 'soil_phh2o_present', 'rain_days_30d', 'rain_7d_CHIRPS', 'rain_90d_CHIRPS', 'ppt7_x_river1000'],
    DRP: ['pet', 'MNDWI_clip99', 'swir16', 'swir22', 'forest_present_2000m', 'forest_pct_2000m', 'river_count_250m', 'river_count_1000m', 'log_dist_river_min_m', 'log_river_len_ratio_250_2000', 'ppt_7d_log1p', 'NDMI_x_river250', 'NDMI_x_river1000', 'ppt7_x_river1000']
}

# GAMs require continuous, strictly non-collinear features. 
GAM_FEATS = {
    TA: ['pet', 'MNDWI_clip99', 'rain_90d_CHIRPS', 'river_count_1000m'],
    EC: ['pet', 'MNDWI_clip99', 'rain_90d_CHIRPS', 'river_count_1000m'],
    DRP: ['pet', 'MNDWI_clip99', 'ppt_7d_log1p', 'log_dist_river_min_m']
}

# 4. FIXED HYPERPARAMETERS (From best_model1.ipynb)
XGB_P = dict(n_estimators=600, learning_rate=0.05, subsample=0.4, colsample_bytree=0.8, max_depth=4, min_child_weight=5, reg_lambda=2.0, random_state=42, n_jobs=-1, verbosity=0)
LGB_P = dict(n_estimators=600, learning_rate=0.05, subsample=0.7, colsample_bytree=0.7, num_leaves=15, min_child_samples=20, reg_lambda=2.0, random_state=42, n_jobs=-1, verbose=-1)
CAT_P = dict(iterations=600, learning_rate=0.05, depth=4, l2_leaf_reg=2.0, subsample=0.3, random_seed=42, verbose=False)

# 5. PRODUCTION PIPELINE: GAM + RESIDUAL BOOSTING
preds_test = {}

print('='*60)
print('TRAINING FINAL GAM-RESIDUAL ENSEMBLE ON FULL DATASET')
print('='*60)

for tgt in TARGETS:
    print(f"\n--- Processing {tgt} ---")
    
    # 5.1 Data Prep for Target
    valid_mask = train[tgt].notna()
    y_train_raw = train.loc[valid_mask, tgt].values
    
    # Impute missing values
    imp_tree = SimpleImputer(strategy='median')
    imp_gam = SimpleImputer(strategy='median')
    
    X_tree_tr = imp_tree.fit_transform(train.loc[valid_mask, TREE_FEATS[tgt]])
    X_tree_te = imp_tree.transform(test[TREE_FEATS[tgt]])
    
    X_gam_tr = imp_gam.fit_transform(train.loc[valid_mask, GAM_FEATS[tgt]])
    X_gam_te = imp_gam.transform(test[GAM_FEATS[tgt]])
    
    # Target Transformation
    is_drp = 'Phosphorus' in tgt
    if is_drp:
        cap = float(np.quantile(y_train_raw, DRP_Q))
        y_train_m = np.log1p(np.minimum(y_train_raw, cap))
    else:
        y_train_m = y_train_raw
        
    # 5.2 Train the GAM (The Macro Model)
    print("Fitting GAM to capture Out-of-Distribution macro-trend...")
    
    # Build spline term dynamically based on number of GAM features
    n_gam_feats = len(GAM_FEATS[tgt])
    spline_term = s(0)
    for i in range(1, n_gam_feats):
        spline_term += s(i)
        
    # NOTE: Dropped the explicit `link='log'` arguments here to prevent the TypeError.
    if tgt == TA:
        gam = GammaGAM(spline_term).fit(X_gam_tr, y_train_m)
    elif tgt == EC:
        gam = InvGaussGAM(spline_term).fit(X_gam_tr, y_train_m)
    else:
        gam = LinearGAM(spline_term).fit(X_gam_tr, y_train_m)

    # 5.3 Calculate Residuals
    y_gam_pred_tr = gam.predict(X_gam_tr)
    y_residuals = y_train_m - y_gam_pred_tr
    
    # 5.4 Train 3-Way Tree Ensemble on Residuals (The Micro Model)
    print(f"Fitting 3-way tree ensemble ({N_SEEDS} seeds) on GAM residuals...")
    xpreds_res, lpreds_res, cpreds_res = [], [], []
    
    for seed in range(N_SEEDS):
        xgb_m = XGBRegressor(**{**XGB_P, 'random_state': seed}).fit(X_tree_tr, y_residuals)
        xpreds_res.append(xgb_m.predict(X_tree_te))
        
        lgb_m = lgb.LGBMRegressor(**{**LGB_P, 'random_state': seed}).fit(X_tree_tr, y_residuals)
        lpreds_res.append(lgb_m.predict(X_tree_te))
        
        cb_m = CatBoostRegressor(**{**CAT_P, 'random_seed': seed}).fit(X_tree_tr, y_residuals)
        cpreds_res.append(cb_m.predict(X_tree_te))

    # Average the tree residual predictions
    ensemble_res_te = (WEIGHTS[0]*np.mean(xpreds_res, 0) + WEIGHTS[1]*np.mean(lpreds_res, 0) + WEIGHTS[2]*np.mean(cpreds_res, 0))
    
    # 5.5 Final Combination: GAM Trend + Tree Residuals
    y_gam_pred_te = gam.predict(X_gam_te)
    final_pred_transformed = y_gam_pred_te + ensemble_res_te
    
    # 5.6 Inverse Transformation and Safety Clipping
    if is_drp:
        final_pred_real = np.expm1(final_pred_transformed)
    else:
        final_pred_real = final_pred_transformed
        
    # Prevent physical impossibilities (concentrations cannot drop below 0)
    preds_test[tgt] = np.maximum(final_pred_real, 0)
    print(f"[{tgt}] COMPLETE - Mean Prediction: {preds_test[tgt].mean():.2f}")

# 6. SAVE SUBMISSION
for tgt in TARGETS:
    tmpl[tgt] = preds_test[tgt]
tmpl.to_csv('submission_gam_residual_boosting.csv', index=False)
print("\nFinished GAM Residual Boosting Pipeline.")